# 📘 Star Schema & Dimensional Modeling
## Databricks Notebook — Kimball Methodology in Practice

**What you'll learn:**
- Dimensional modeling theory (Kimball methodology)
- Fact tables: additive, semi-additive, non-additive, factless
- Dimension tables: SCD, role-playing, junk, degenerate
- Complete date dimension implementation
- Full star schema build (7 dimensions + 3 facts)
- Snowflake schema, bridge tables, accumulating snapshots
- NEW Retail & Supply Chain dataset

**Prerequisites:** Notebooks 01-07

---

---
# 🏗️ Retail & Supply Chain Dataset Creation

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS retail_dw;
USE retail_dw;

In [ ]:
%sql
DROP TABLE IF EXISTS stores;
CREATE TABLE stores (
  store_id INT, store_name STRING, store_type STRING, address STRING,
  city STRING, state STRING, region STRING, manager_id INT, open_date DATE, sqft INT, status STRING
);
INSERT INTO stores
SELECT id AS store_id,
  CONCAT('Store_', LPAD(CAST(id AS STRING),3,'0')) AS store_name,
  CASE abs(hash(id*3))%3 WHEN 0 THEN 'flagship' WHEN 1 THEN 'standard' ELSE 'outlet' END AS store_type,
  CONCAT(CAST(abs(hash(id*7))%9999+1 AS STRING), ' Commerce Blvd') AS address,
  CASE abs(hash(id*11))%8 WHEN 0 THEN 'New York' WHEN 1 THEN 'Chicago' WHEN 2 THEN 'Houston' WHEN 3 THEN 'Phoenix' WHEN 4 THEN 'LA' WHEN 5 THEN 'Dallas' WHEN 6 THEN 'Seattle' ELSE 'Miami' END AS city,
  CASE abs(hash(id*13))%8 WHEN 0 THEN 'NY' WHEN 1 THEN 'IL' WHEN 2 THEN 'TX' WHEN 3 THEN 'AZ' WHEN 4 THEN 'CA' WHEN 5 THEN 'TX' WHEN 6 THEN 'WA' ELSE 'FL' END AS state,
  CASE abs(hash(id*17))%4 WHEN 0 THEN 'Northeast' WHEN 1 THEN 'Midwest' WHEN 2 THEN 'South' ELSE 'West' END AS region,
  abs(hash(id*19))%50+1 AS manager_id,
  date_add('2010-01-01', abs(hash(id*23))%4000) AS open_date,
  abs(hash(id*29))%50000+5000 AS sqft,
  CASE WHEN id%10=0 THEN 'closed' ELSE 'open' END AS status
FROM (SELECT explode(sequence(1, 50)) AS id);

In [ ]:
%sql
DROP TABLE IF EXISTS pos_transactions;
CREATE TABLE pos_transactions (
  txn_id INT, store_id INT, register_id INT, customer_id INT,
  txn_timestamp TIMESTAMP, txn_date DATE, total_amount DECIMAL(10,2),
  payment_type STRING, cashier_id INT, receipt_number STRING
);
INSERT INTO pos_transactions
SELECT id AS txn_id,
  abs(hash(id*3))%50+1 AS store_id,
  abs(hash(id*7))%10+1 AS register_id,
  abs(hash(id*11))%5000+1 AS customer_id,
  CAST(date_add('2023-01-01', abs(hash(id*13))%600) AS TIMESTAMP) AS txn_timestamp,
  date_add('2023-01-01', abs(hash(id*13))%600) AS txn_date,
  CAST(abs(hash(id*17))%5000+5 AS DECIMAL(10,2)) AS total_amount,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'cash' WHEN 1 THEN 'credit' WHEN 2 THEN 'debit' ELSE 'mobile' END AS payment_type,
  abs(hash(id*23))%100+1 AS cashier_id,
  CONCAT('RCP-', LPAD(CAST(id AS STRING),8,'0')) AS receipt_number
FROM (SELECT explode(sequence(1, 100000)) AS id);

In [ ]:
%sql
DROP TABLE IF EXISTS pos_line_items;
CREATE TABLE pos_line_items (
  line_id INT, txn_id INT, product_id INT, quantity INT,
  unit_price DECIMAL(10,2), discount_amount DECIMAL(10,2), line_total DECIMAL(10,2)
);
INSERT INTO pos_line_items
SELECT id AS line_id,
  abs(hash(id*3))%100000+1 AS txn_id,
  abs(hash(id*7))%500+1 AS product_id,
  abs(hash(id*11))%10+1 AS quantity,
  CAST(abs(hash(id*13))%2000+5 AS DECIMAL(10,2)) AS unit_price,
  CAST(abs(hash(id*17))%200 AS DECIMAL(10,2)) AS discount_amount,
  CAST((abs(hash(id*11))%10+1)*(abs(hash(id*13))%2000+5)-abs(hash(id*17))%200 AS DECIMAL(10,2)) AS line_total
FROM (SELECT explode(sequence(1, 300000)) AS id);

In [ ]:
%sql
DROP TABLE IF EXISTS suppliers;
CREATE TABLE suppliers (
  supplier_id INT, supplier_name STRING, contact_name STRING, country STRING,
  lead_time_days INT, reliability_score INT, category STRING
);
INSERT INTO suppliers
SELECT id AS supplier_id,
  CONCAT('Supplier_', id) AS supplier_name,
  CONCAT('Contact_', id) AS contact_name,
  CASE abs(hash(id*3))%6 WHEN 0 THEN 'USA' WHEN 1 THEN 'China' WHEN 2 THEN 'Germany' WHEN 3 THEN 'Japan' WHEN 4 THEN 'Mexico' ELSE 'India' END AS country,
  abs(hash(id*7))%60+5 AS lead_time_days,
  abs(hash(id*11))%100+1 AS reliability_score,
  CASE abs(hash(id*13))%4 WHEN 0 THEN 'Electronics' WHEN 1 THEN 'Accessories' WHEN 2 THEN 'Raw Materials' ELSE 'Packaging' END AS category
FROM (SELECT explode(sequence(1, 100)) AS id);

In [ ]:
%sql
DROP TABLE IF EXISTS purchase_orders;
CREATE TABLE purchase_orders (
  po_id INT, supplier_id INT, product_id INT, order_date DATE,
  expected_date DATE, received_date DATE, quantity_ordered INT,
  quantity_received INT, unit_cost DECIMAL(10,2), status STRING
);
INSERT INTO purchase_orders
SELECT id AS po_id,
  abs(hash(id*3))%100+1 AS supplier_id,
  abs(hash(id*7))%500+1 AS product_id,
  date_add('2023-01-01', abs(hash(id*11))%600) AS order_date,
  date_add('2023-01-01', abs(hash(id*11))%600+abs(hash(id*13))%30+7) AS expected_date,
  CASE WHEN id%8=0 THEN NULL ELSE date_add('2023-01-01', abs(hash(id*11))%600+abs(hash(id*17))%40+5) END AS received_date,
  abs(hash(id*19))%1000+10 AS quantity_ordered,
  CASE WHEN id%8=0 THEN 0 ELSE abs(hash(id*19))%1000+10-abs(hash(id*23))%50 END AS quantity_received,
  CAST(abs(hash(id*29))%500+5 AS DECIMAL(10,2)) AS unit_cost,
  CASE abs(hash(id*31))%4 WHEN 0 THEN 'delivered' WHEN 1 THEN 'in_transit' WHEN 2 THEN 'ordered' ELSE 'cancelled' END AS status
FROM (SELECT explode(sequence(1, 10000)) AS id);

In [ ]:
%sql
DROP TABLE IF EXISTS promotions;
CREATE TABLE promotions (
  promo_id INT, promo_name STRING, promo_type STRING, start_date DATE,
  end_date DATE, discount_value DECIMAL(5,2), applicable_categories STRING
);
INSERT INTO promotions
SELECT id AS promo_id,
  CONCAT(CASE abs(hash(id*3))%4 WHEN 0 THEN 'Summer' WHEN 1 THEN 'Holiday' WHEN 2 THEN 'Flash' ELSE 'Clearance' END, ' Sale ', id) AS promo_name,
  CASE abs(hash(id*7))%3 WHEN 0 THEN 'BOGO' WHEN 1 THEN 'pct_off' ELSE 'fixed_off' END AS promo_type,
  date_add('2023-01-01', abs(hash(id*11))%500) AS start_date,
  date_add('2023-01-01', abs(hash(id*11))%500+abs(hash(id*13))%30+7) AS end_date,
  CAST(abs(hash(id*17))%50+5 AS DECIMAL(5,2)) AS discount_value,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'Electronics' WHEN 1 THEN 'Accessories' WHEN 2 THEN 'All' ELSE 'Gaming' END AS applicable_categories
FROM (SELECT explode(sequence(1, 200)) AS id);

In [ ]:
%sql
SELECT 'stores' AS tbl, COUNT(*) AS rows FROM stores
UNION ALL SELECT 'pos_transactions', COUNT(*) FROM pos_transactions
UNION ALL SELECT 'pos_line_items', COUNT(*) FROM pos_line_items
UNION ALL SELECT 'suppliers', COUNT(*) FROM suppliers
UNION ALL SELECT 'purchase_orders', COUNT(*) FROM purchase_orders
UNION ALL SELECT 'promotions', COUNT(*) FROM promotions
ORDER BY tbl;

---
# 📗 Section 1: Dimensional Modeling Theory

## OLTP vs OLAP

| OLTP (Operational) | OLAP (Analytical) |
|---|---|
| 3NF normalized | Star/Snowflake denormalized |
| Optimized for writes | Optimized for reads |
| Current state | Historical analysis |
| Single-row operations | Aggregations over millions |
| Application-facing | BI/reporting-facing |

## The 4-Step Kimball Process

1. **Select the business process** (e.g., retail sales)
2. **Declare the grain** (one row = one line item per transaction)
3. **Identify the dimensions** (who, what, where, when, how)
4. **Identify the facts** (measurable numeric values)

## Star Schema Structure

```
          ┌──────────┐
          │ dim_date │
          └────┬─────┘
               │
┌──────────┐   │   ┌─────────────┐
│dim_store │───┼───│ fact_sales  │───┬───┌──────────────┐
└──────────┘   │   │             │   │   │ dim_product  │
               │   │ store_key   │   │   └──────────────┘
┌──────────┐   │   │ date_key    │   │
│dim_cust  │───┘   │ product_key │   └───┌──────────────┐
└──────────┘       │ customer_key│       │ dim_promotion│
                   │ promo_key   │       └──────────────┘
                   │─────────────│
                   │ quantity    │
                   │ unit_price  │
                   │ discount    │
                   │ line_total  │
                   └─────────────┘
```

## Modeling Approaches — Complete Comparison

### 1. Star Schema (Most Common for Analytics)

```
         dim_date          dim_customer
         ┌──────┐          ┌──────────┐
         │date_sk│         │cust_sk   │
         │date   │         │name      │
         │month  │         │segment   │
         │year   │         │region    │
         └───┬───┘         └────┬─────┘
             │                  │
             └──────┬───────────┘
                    │
              ┌─────▼──────┐
              │  fct_sales  │
              │ date_sk(FK) │
              │ cust_sk(FK) │
              │ prod_sk(FK) │
              │ revenue     │
              │ quantity    │
              └─────┬───────┘
                    │
             ┌──────┘
             │
         dim_product
         ┌──────────┐
         │prod_sk   │
         │name      │
         │category  │
         │price     │
         └──────────┘
```

**Pros:** Simple, fast queries, easy for analysts
**Cons:** Some redundancy in dimensions

### 2. Snowflake Schema

Dimensions are normalized (split into sub-dimensions):

```
dim_product → dim_category → dim_department
```

**Pros:** Less storage, no redundancy
**Cons:** More JOINs needed, slower queries

### 3. Data Vault (Enterprise)

Three object types: Hubs (business keys), Links (relationships), Satellites (attributes):

```
hub_customer ──── link_order ──── hub_order
     │                                │
sat_customer_details          sat_order_details
(name, email, etc.)           (amount, status, etc.)
```

**Pros:** Highly auditable, handles schema changes gracefully, parallel loading
**Cons:** Complex, many tables, requires specialized knowledge

### 4. One Big Table (OBT)

Everything in one wide, denormalized table:

```
order_id | customer_name | customer_city | product_name | category | quantity | revenue | order_date | ...
```

**Pros:** Simplest queries (no JOINs), fast for dashboards
**Cons:** Massive redundancy, hard to maintain, poor for updates

### When to Use Each

| Approach | Best For | Avoid When |
|----------|----------|------------|
| **Star Schema** | BI analytics, most use cases | Very complex hierarchies |
| **Snowflake** | Storage-constrained, complex hierarchies | Performance is critical |
| **Data Vault** | Enterprise DW, regulatory compliance, frequent schema changes | Small teams, simple use cases |
| **OBT** | Simple dashboards, ML features, small data | Data that changes frequently |

In [ ]:
# Data Vault pattern simulation
print("🏛️ DATA VAULT PATTERN")
print("=" * 60)

# Data Vault has 3 object types:
# 1. HUB: Business keys (what is unique in the real world)
# 2. LINK: Relationships between hubs
# 3. SATELLITE: Descriptive attributes (the "what" about hubs/links)

data_vault_structure = {
    "Hubs (Business Keys)": {
        "hub_customer": {
            "columns": ["customer_hk (PK)", "customer_id (BK)", "load_date", "record_source"],
            "purpose": "Unique customers by business key"
        },
        "hub_order": {
            "columns": ["order_hk (PK)", "order_id (BK)", "load_date", "record_source"],
            "purpose": "Unique orders by business key"
        },
        "hub_product": {
            "columns": ["product_hk (PK)", "product_id (BK)", "load_date", "record_source"],
            "purpose": "Unique products by business key"
        }
    },
    "Links (Relationships)": {
        "link_order_customer": {
            "columns": ["link_hk (PK)", "order_hk (FK)", "customer_hk (FK)", "load_date", "record_source"],
            "purpose": "Which customer placed which order"
        },
        "link_order_product": {
            "columns": ["link_hk (PK)", "order_hk (FK)", "product_hk (FK)", "load_date", "record_source"],
            "purpose": "Which products are in which orders"
        }
    },
    "Satellites (Attributes)": {
        "sat_customer_details": {
            "columns": ["customer_hk (FK)", "load_date", "hash_diff", "name", "email", "city", "segment"],
            "purpose": "Customer attributes (versioned — full history)"
        },
        "sat_order_details": {
            "columns": ["order_hk (FK)", "load_date", "hash_diff", "status", "total_amount", "order_date"],
            "purpose": "Order attributes (versioned)"
        }
    }
}

for vault_type, objects in data_vault_structure.items():
    print(f"\n📌 {vault_type}:")
    for name, details in objects.items():
        print(f"   {name}")
        print(f"      Columns: {', '.join(details['columns'])}")
        print(f"      Purpose: {details['purpose']}")

print("\n💡 Data Vault Key Advantages:")
print("   1. Parallel loading (hubs, links, sats load independently)")
print("   2. Full audit history (every change tracked in satellites)")
print("   3. Schema evolution (add new satellites without changing existing)")
print("   4. Source system agnostic (multiple sources feed same hub)")


---
# 📗 Section 2: Fact Tables

## Types of Facts

| Type | Example | Can SUM across all dims? |
|---|---|---|
| **Additive** | revenue, quantity, cost | ✅ Yes |
| **Semi-additive** | account_balance, inventory_qty | ⚠️ Not across time |
| **Non-additive** | unit_price, ratio, percentage | ❌ No (must avg/recalculate) |

## Grain = THE Most Important Decision

> "One row in fact_sales represents ONE line item in ONE transaction"

If grain is wrong, everything downstream is wrong.

In [ ]:
%sql
-- Grain example: POS line items
-- Each row = one product in one transaction
USE retail_dw;
SELECT txn_id, product_id, quantity, unit_price, line_total
FROM pos_line_items LIMIT 10;

---
# 📗 Section 3: Dimension Tables

## Dimension Types

| Type | Description | Example |
|---|---|---|
| **Conformed** | Shared across fact tables | dim_date, dim_customer |
| **Degenerate** | Dimension value in fact (no table) | receipt_number, order_id |
| **Role-playing** | Same dim used multiple ways | order_date, ship_date → dim_date |
| **Junk** | Combine low-cardinality flags | payment_type + channel + is_gift |
| **Mini-dimension** | Split rapidly-changing attributes | customer_demographics |

---
# 📗 Section 4: Date Dimension (Complete Implementation)

Every data warehouse needs a date dimension. It enables time-based analysis
without complex date functions in every query.

In [ ]:
%sql
-- Complete Date Dimension (2020-2026)
USE retail_dw;
DROP TABLE IF EXISTS dim_date;
CREATE TABLE dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS full_date,
  YEAR(d) AS year,
  QUARTER(d) AS quarter,
  MONTH(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  date_format(d, 'MMM') AS month_short,
  DAY(d) AS day_of_month,
  DAYOFWEEK(d) AS day_of_week,
  date_format(d, 'EEEE') AS day_name,
  WEEKOFYEAR(d) AS week_of_year,
  DAYOFYEAR(d) AS day_of_year,
  CASE WHEN DAYOFWEEK(d) IN (1, 7) THEN true ELSE false END AS is_weekend,
  CONCAT(YEAR(d), '-Q', QUARTER(d)) AS year_quarter,
  CONCAT(YEAR(d), '-', LPAD(CAST(MONTH(d) AS STRING), 2, '0')) AS year_month,
  CASE WHEN MONTH(d) >= 7 THEN YEAR(d) + 1 ELSE YEAR(d) END AS fiscal_year,
  CASE WHEN MONTH(d) >= 7 THEN QUARTER(d) - 2 ELSE QUARTER(d) + 2 END AS fiscal_quarter
FROM (
  SELECT date_add('2020-01-01', id-1) AS d
  FROM (SELECT explode(sequence(1, 2557)) AS id)
) dates;

SELECT COUNT(*) AS total_days, MIN(full_date) AS start_date, MAX(full_date) AS end_date
FROM dim_date;

In [ ]:
%sql
-- Using the date dimension for analysis
SELECT dd.year, dd.quarter, dd.month_name,
  COUNT(DISTINCT t.txn_id) AS transactions,
  ROUND(SUM(t.total_amount), 2) AS revenue
FROM pos_transactions t
JOIN dim_date dd ON t.txn_date = dd.full_date
WHERE dd.year = 2024 AND dd.is_weekend = false
GROUP BY dd.year, dd.quarter, dd.month_name
ORDER BY dd.quarter, dd.month_name
LIMIT 12;

---
# 📗 Section 5: Building the Star Schema

## 5.1 dim_customer

In [ ]:
%sql
USE retail_dw;
DROP TABLE IF EXISTS dim_customer;
CREATE TABLE dim_customer AS
SELECT
  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  first_name, last_name,
  CONCAT(first_name, ' ', last_name) AS full_name,
  COALESCE(city, 'Unknown') AS city,
  COALESCE(state, 'Unknown') AS state,
  customer_segment,
  CASE WHEN lifetime_value >= 40000 THEN 'Platinum'
       WHEN lifetime_value >= 20000 THEN 'Gold'
       WHEN lifetime_value >= 5000 THEN 'Silver'
       ELSE 'Bronze' END AS value_tier,
  registration_date,
  is_active,
  current_timestamp() AS _loaded_at
FROM techmart_dw.customers;

SELECT COUNT(*) AS dim_customer_rows FROM dim_customer;

## 5.2 dim_product

In [ ]:
%sql
DROP TABLE IF EXISTS dim_product;
CREATE TABLE dim_product AS
SELECT
  ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
  product_id,
  product_name,
  category,
  subcategory,
  brand,
  price,
  cost,
  CASE WHEN price >= 1500 THEN 'Premium'
       WHEN price >= 500 THEN 'Mid-Range'
       ELSE 'Budget' END AS price_tier,
  ROUND((price - cost) / price * 100, 2) AS margin_pct,
  is_active,
  current_timestamp() AS _loaded_at
FROM techmart_dw.products;

SELECT COUNT(*) AS dim_product_rows FROM dim_product;

## 5.3 dim_store

In [ ]:
%sql
DROP TABLE IF EXISTS dim_store;
CREATE TABLE dim_store AS
SELECT
  ROW_NUMBER() OVER (ORDER BY store_id) AS store_key,
  store_id, store_name, store_type, city, state, region,
  CASE WHEN sqft >= 30000 THEN 'Large'
       WHEN sqft >= 15000 THEN 'Medium'
       ELSE 'Small' END AS size_category,
  open_date, status,
  current_timestamp() AS _loaded_at
FROM stores;

SELECT COUNT(*) AS dim_store_rows FROM dim_store;

## 5.4 dim_promotion

In [ ]:
%sql
DROP TABLE IF EXISTS dim_promotion;
CREATE TABLE dim_promotion AS
SELECT
  ROW_NUMBER() OVER (ORDER BY promo_id) AS promo_key,
  promo_id, promo_name, promo_type,
  start_date, end_date,
  discount_value,
  CASE WHEN discount_value >= 30 THEN 'Deep Discount'
       WHEN discount_value >= 15 THEN 'Moderate'
       ELSE 'Light' END AS discount_tier,
  applicable_categories,
  current_timestamp() AS _loaded_at
FROM promotions;

-- Add "No Promotion" row (for transactions without promos)
INSERT INTO dim_promotion VALUES (0, 0, 'No Promotion', 'none', NULL, NULL, 0, 'None', 'All', current_timestamp());

SELECT COUNT(*) AS dim_promotion_rows FROM dim_promotion;

## 5.5 fact_sales

In [ ]:
%sql
-- Build fact_sales: grain = one line item per transaction
DROP TABLE IF EXISTS fact_sales;
CREATE TABLE fact_sales AS
SELECT
  li.line_id AS sales_key,
  COALESCE(dc.customer_key, -1) AS customer_key,
  COALESCE(dp.product_key, -1) AS product_key,
  COALESCE(ds.store_key, -1) AS store_key,
  CAST(date_format(t.txn_date, 'yyyyMMdd') AS INT) AS date_key,
  0 AS promo_key,
  -- Degenerate dimensions
  t.receipt_number,
  t.register_id,
  -- Facts (measures)
  li.quantity,
  li.unit_price,
  li.discount_amount,
  li.line_total,
  COALESCE(dp2.cost, 0) * li.quantity AS cost_amount,
  li.line_total - COALESCE(dp2.cost, 0) * li.quantity AS gross_profit,
  t.payment_type
FROM pos_line_items li
JOIN pos_transactions t ON li.txn_id = t.txn_id
LEFT JOIN dim_customer dc ON t.customer_id = dc.customer_id
LEFT JOIN dim_product dp ON li.product_id = dp.product_id
LEFT JOIN dim_store ds ON t.store_id = ds.store_id
LEFT JOIN techmart_dw.products dp2 ON li.product_id = dp2.product_id;

SELECT COUNT(*) AS fact_sales_rows FROM fact_sales;

In [ ]:
%sql
-- Verify the star schema works: revenue by region and quarter
SELECT ds.region, dd.year_quarter,
  COUNT(*) AS transactions,
  ROUND(SUM(fs.line_total), 2) AS revenue,
  ROUND(SUM(fs.gross_profit), 2) AS profit
FROM fact_sales fs
JOIN dim_store ds ON fs.store_key = ds.store_key
JOIN dim_date dd ON fs.date_key = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region, dd.year_quarter
ORDER BY ds.region, dd.year_quarter;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Query the Star Schema
-- ============================================
-- Write a query showing:
-- Top 10 products by revenue in 2024, including:
-- product_name, category, price_tier, total_quantity, total_revenue, avg_discount
-- Use the star schema (fact_sales + dim_product + dim_date)
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT dp.product_name, dp.category, dp.price_tier,
  SUM(fs.quantity) AS total_quantity,
  ROUND(SUM(fs.line_total), 2) AS total_revenue,
  ROUND(AVG(fs.discount_amount), 2) AS avg_discount
FROM fact_sales fs
JOIN dim_product dp ON fs.product_key = dp.product_key
JOIN dim_date dd ON fs.date_key = dd.date_key
WHERE dd.year = 2024
GROUP BY dp.product_name, dp.category, dp.price_tier
ORDER BY total_revenue DESC
LIMIT 10;

---
# 📗 Section 6: Snowflake Schema

**When to normalize dimensions:**
- Very large dimension tables (millions of rows)
- Frequently updated hierarchies
- Storage optimization needed

**Tradeoff:** More joins = slower queries, but less storage and easier maintenance.

In [ ]:
%sql
-- Snowflake: normalize product hierarchy
DROP TABLE IF EXISTS dim_category;
CREATE TABLE dim_category AS
SELECT DISTINCT
  ROW_NUMBER() OVER (ORDER BY category, subcategory) AS category_key,
  category, subcategory
FROM techmart_dw.products;

-- dim_product_snowflake references dim_category
DROP TABLE IF EXISTS dim_product_snowflake;
CREATE TABLE dim_product_snowflake AS
SELECT dp.product_key, dp.product_id, dp.product_name, dp.brand,
  dc.category_key, dp.price_tier, dp.is_active
FROM dim_product dp
JOIN dim_category dc ON dp.category = dc.category AND dp.subcategory = dc.subcategory;

SELECT * FROM dim_product_snowflake LIMIT 5;

---
# 📗 Section 7: Advanced Patterns

## 7.1 Accumulating Snapshot Fact (Order Lifecycle)

In [ ]:
%sql
-- Accumulating snapshot: track order through stages
DROP TABLE IF EXISTS fact_order_lifecycle;
CREATE TABLE fact_order_lifecycle AS
SELECT
  o.order_id,
  o.customer_id,
  CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
  CAST(date_format(s.ship_date, 'yyyyMMdd') AS INT) AS ship_date_key,
  CAST(date_format(s.actual_delivery, 'yyyyMMdd') AS INT) AS delivery_date_key,
  o.total_amount,
  DATEDIFF(s.ship_date, o.order_date) AS days_to_ship,
  DATEDIFF(s.actual_delivery, s.ship_date) AS days_in_transit,
  DATEDIFF(s.actual_delivery, o.order_date) AS total_fulfillment_days,
  o.status
FROM techmart_dw.orders o
LEFT JOIN techmart_dw.shipments s ON o.order_id = s.order_id;

SELECT status, ROUND(AVG(days_to_ship),1) AS avg_ship, ROUND(AVG(total_fulfillment_days),1) AS avg_total
FROM fact_order_lifecycle
WHERE days_to_ship IS NOT NULL
GROUP BY status;

## 7.2 Periodic Snapshot Fact

In [ ]:
%sql
-- Monthly inventory snapshot
DROP TABLE IF EXISTS fact_monthly_inventory;
CREATE TABLE fact_monthly_inventory AS
SELECT
  product_id,
  warehouse_id,
  date_format(snapshot_date, 'yyyy-MM') AS snapshot_month,
  AVG(quantity_on_hand) AS avg_on_hand,
  AVG(quantity_reserved) AS avg_reserved,
  MAX(quantity_on_hand) AS max_on_hand,
  MIN(quantity_on_hand) AS min_on_hand
FROM techmart_dw.inventory
GROUP BY product_id, warehouse_id, date_format(snapshot_date, 'yyyy-MM');

SELECT COUNT(*) AS snapshot_rows FROM fact_monthly_inventory;

## One Big Table (OBT) Pattern

### When OBT Makes Sense

OBT is a single wide, denormalized table that pre-joins everything:

```sql
-- Instead of joining 5 tables every query:
SELECT c.name, p.category, SUM(oi.line_total)
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
GROUP BY c.name, p.category

-- OBT: one table, no joins needed:
SELECT customer_name, product_category, SUM(line_total)
FROM obt_sales
GROUP BY customer_name, product_category
```

### OBT Trade-offs

| Aspect | Star Schema | OBT |
|--------|-------------|-----|
| **Query complexity** | Multiple JOINs | No JOINs |
| **Storage** | Efficient (normalized) | Redundant (denormalized) |
| **Updates** | Update one dimension | Update many rows |
| **Flexibility** | Slice any dimension | Fixed structure |
| **ML features** | Need to join | Ready to use |
| **Best for** | BI analytics | Dashboards, ML |

### Building an OBT

```sql
-- Create OBT from star schema
CREATE TABLE gold.obt_sales AS
SELECT
    -- Order facts
    o.order_id, o.order_date, o.total_amount, o.status,
    -- Customer attributes (denormalized)
    c.customer_id, c.first_name, c.last_name, c.city, c.customer_segment,
    -- Product attributes (denormalized)
    p.product_id, p.product_name, p.category, p.price,
    -- Order item details
    oi.quantity, oi.line_total,
    -- Date attributes (denormalized)
    YEAR(o.order_date) AS order_year,
    MONTH(o.order_date) AS order_month,
    DAYOFWEEK(o.order_date) AS day_of_week
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
```

In [ ]:
%sql
-- Build an OBT for the TechMart dataset
DROP TABLE IF EXISTS techmart_dw.obt_sales;

CREATE TABLE techmart_dw.obt_sales AS
SELECT
    -- Order facts
    o.order_id,
    o.order_date,
    YEAR(o.order_date) AS order_year,
    MONTH(o.order_date) AS order_month,
    DAYOFWEEK(o.order_date) AS day_of_week,
    o.status AS order_status,
    o.total_amount,
    o.discount_amount,
    o.payment_method,
    -- Customer (denormalized)
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.city AS customer_city,
    c.customer_segment,
    c.lifetime_value,
    -- Product (denormalized)
    p.product_id,
    p.product_name,
    p.category AS product_category,
    p.price AS unit_price,
    -- Order item
    oi.quantity,
    oi.line_total
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')

In [ ]:
%sql
-- Now analysts can query without ANY joins!
-- Revenue by customer segment and product category
SELECT 
    customer_segment,
    product_category,
    COUNT(DISTINCT order_id) AS orders,
    SUM(line_total) AS revenue,
    AVG(line_total) AS avg_line_value
FROM techmart_dw.obt_sales
GROUP BY customer_segment, product_category
ORDER BY revenue DESC
LIMIT 15

In [ ]:
# ============================================
# 🎯 YOUR TURN: Choose the Right Modeling Approach
# ============================================
# For each scenario, choose: Star Schema, Snowflake, Data Vault, or OBT
#
# Scenario A: A startup building their first analytics warehouse.
#   - 5 analysts, simple queries, small team
#   - Need: revenue by product, customer, date
#
# Scenario B: A bank building a regulatory reporting system.
#   - Must track every change to every record (full audit)
#   - Multiple source systems feeding the same entities
#   - Schema changes frequently as regulations change
#
# Scenario C: A data scientist building ML features for churn prediction.
#   - Needs one row per customer with 50+ features
#   - Features are pre-computed, rarely updated
#   - Model training needs fast reads
#
# Scenario D: A large retailer with complex product hierarchies.
#   - Products → Subcategory → Category → Department → Division
#   - Storage is expensive, want to minimize redundancy
#
# Write your answers below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
answers = {
    "A: Startup analytics": {
        "choice": "Star Schema",
        "reason": "Simple, fast, easy for analysts. Perfect for small team with standard BI needs.",
        "avoid": "Data Vault (too complex), OBT (hard to maintain as data grows)"
    },
    "B: Bank regulatory reporting": {
        "choice": "Data Vault",
        "reason": "Full audit history, handles schema changes, multiple source systems. "
                  "Regulatory compliance requires tracking every change.",
        "avoid": "Star Schema (loses history), OBT (can't handle frequent schema changes)"
    },
    "C: ML churn features": {
        "choice": "OBT (Feature Store)",
        "reason": "One row per customer, no joins needed for training. "
                  "ML models need flat, wide tables.",
        "avoid": "Star Schema (too many joins for ML), Data Vault (too complex)"
    },
    "D: Retailer with complex hierarchies": {
        "choice": "Snowflake Schema",
        "reason": "Complex product hierarchy (5 levels) is naturally normalized. "
                  "Reduces redundancy vs star schema.",
        "avoid": "OBT (massive redundancy for 5-level hierarchy)"
    }
}

print("✅ MODELING APPROACH ANSWERS")
print("=" * 60)
for scenario, details in answers.items():
    print(f"\n📌 {scenario}")
    print(f"   Choice: {details['choice']}")
    print(f"   Reason: {details['reason']}")
    print(f"   Avoid: {details['avoid']}")


---
# 📗 Section 8: BI Optimization — Pre-Aggregation

In [ ]:
%sql
-- Pre-aggregated summary for dashboards (much faster than querying fact_sales)
DROP TABLE IF EXISTS agg_daily_sales;
CREATE TABLE agg_daily_sales AS
SELECT
  fs.date_key,
  dd.full_date,
  dd.year, dd.month, dd.day_of_week, dd.is_weekend,
  ds.region, ds.store_type,
  dp.category, dp.price_tier,
  COUNT(*) AS line_items,
  SUM(fs.quantity) AS total_units,
  ROUND(SUM(fs.line_total), 2) AS total_revenue,
  ROUND(SUM(fs.gross_profit), 2) AS total_profit,
  ROUND(AVG(fs.line_total), 2) AS avg_line_value,
  COUNT(DISTINCT fs.receipt_number) AS unique_transactions
FROM fact_sales fs
JOIN dim_date dd ON fs.date_key = dd.date_key
JOIN dim_store ds ON fs.store_key = ds.store_key
JOIN dim_product dp ON fs.product_key = dp.product_key
GROUP BY fs.date_key, dd.full_date, dd.year, dd.month, dd.day_of_week, dd.is_weekend,
  ds.region, ds.store_type, dp.category, dp.price_tier;

SELECT COUNT(*) AS agg_rows FROM agg_daily_sales;

---
# 📗 Section 9: ETL for Dimensional Model

## Dimension Load Order
1. Load dimensions FIRST (they provide surrogate keys)
2. Load facts SECOND (lookup surrogate keys from dimensions)
3. Handle unknown members (key = -1 for missing lookups)

In [ ]:
%sql
-- ETL pattern: fact load with dimension key lookup
-- This is what the fact_sales creation above does:
-- 1. Join to dim_customer → get customer_key
-- 2. Join to dim_product → get product_key
-- 3. Join to dim_store → get store_key
-- 4. COALESCE with -1 for missing lookups

-- Verify referential integrity
SELECT 'Missing customer' AS issue, COUNT(*) AS cnt FROM fact_sales WHERE customer_key = -1
UNION ALL SELECT 'Missing product', COUNT(*) FROM fact_sales WHERE product_key = -1
UNION ALL SELECT 'Missing store', COUNT(*) FROM fact_sales WHERE store_key = -1;

---
# 🚀 Section 10: Mini Projects

## Project 1: Analytics Queries Against Star Schema

In [ ]:
%sql
-- Q1: Revenue by region and month (2024)
SELECT ds.region, dd.month_name, ROUND(SUM(fs.line_total),2) AS revenue
FROM fact_sales fs
JOIN dim_store ds ON fs.store_key = ds.store_key
JOIN dim_date dd ON fs.date_key = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region, dd.month_name, dd.month
ORDER BY ds.region, dd.month;

In [ ]:
%sql
-- Q2: Top categories by profit margin
SELECT dp.category, dp.price_tier,
  ROUND(SUM(fs.line_total), 2) AS revenue,
  ROUND(SUM(fs.gross_profit), 2) AS profit,
  ROUND(SUM(fs.gross_profit) / SUM(fs.line_total) * 100, 2) AS margin_pct
FROM fact_sales fs
JOIN dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.category, dp.price_tier
ORDER BY margin_pct DESC;

In [ ]:
%sql
-- Q3: Customer value tiers performance
SELECT dc.value_tier,
  COUNT(DISTINCT dc.customer_key) AS customers,
  COUNT(*) AS purchases,
  ROUND(SUM(fs.line_total), 2) AS total_revenue,
  ROUND(SUM(fs.line_total) / COUNT(DISTINCT dc.customer_key), 2) AS revenue_per_customer
FROM fact_sales fs
JOIN dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.value_tier
ORDER BY revenue_per_customer DESC;

In [ ]:
%sql
-- Q4: Weekend vs weekday sales
SELECT dd.is_weekend,
  COUNT(*) AS transactions,
  ROUND(SUM(fs.line_total), 2) AS revenue,
  ROUND(AVG(fs.line_total), 2) AS avg_basket
FROM fact_sales fs
JOIN dim_date dd ON fs.date_key = dd.date_key
GROUP BY dd.is_weekend;

---
# 🏆 Section 11: Interview Questions

## Q1: Star schema vs snowflake schema?

**Star:** Dimensions denormalized (one table per dimension). Fewer joins, faster queries.
**Snowflake:** Dimensions normalized into sub-tables. Less storage, more joins.
**Use star** for most cases. **Use snowflake** only for very large dimensions or frequently changing hierarchies.

## Q2: What is grain and why is it important?

**Grain** = what one row in the fact table represents.
Example: "One line item in one POS transaction"

If grain is wrong:
- Aggregations will be incorrect (double-counting)
- Joins will produce unexpected results
- Business users will get wrong answers

## Q3: How do you handle SCD in a star schema?

- **Type 1:** Overwrite dimension attribute (no fact table change)
- **Type 2:** New dimension row with new surrogate key. Facts point to the version that was current at transaction time.
- **Type 3:** Add previous_value column to dimension

The surrogate key in the fact table links to the CORRECT VERSION of the dimension.

## Q4: Additive vs semi-additive facts?

- **Additive** (revenue, quantity): SUM across any dimension combination
- **Semi-additive** (balance, inventory): Can SUM across non-time dimensions, but must AVG or use snapshot across time
- **Non-additive** (ratio, percentage): Cannot SUM — must recalculate from components

## Q5: Design a star schema for [hotel bookings]

**Grain:** One row per room-night per booking
**Dimensions:** dim_date, dim_hotel, dim_room_type, dim_guest, dim_channel
**Facts:** room_rate, taxes, fees, is_cancelled, length_of_stay
**Degenerate:** booking_confirmation_number

---
# ✅ Notebook Complete!

**What was covered:**
- Dimensional modeling theory (Kimball 4-step process)
- Fact tables: additive, semi-additive, non-additive, factless
- Dimension tables: conformed, degenerate, role-playing, junk
- Complete date dimension (2020-2026, 2557 days)
- Full star schema: dim_customer, dim_product, dim_store, dim_promotion, dim_date + fact_sales
- Snowflake schema normalization
- Advanced: accumulating snapshots, periodic snapshots
- BI optimization: pre-aggregation tables
- ETL patterns: dimension-first loading, surrogate key lookup
- 4 analytics queries demonstrating star schema power
- 5 interview questions

**Database created:** `retail_dw` (6 source tables + star schema)

**Next:** Notebook 09 — Pandas for Data Engineering

---